# Ayudantía 07: Threading 🧵
Por sus queridos ayudantes de catedra:

* Diego Toledo
* Francisca Cares
* Carlos Martel
* Agustín Becker
* Julio Huerta

### Hoy veremos
1. Threads 
2. Locks
3. Eventos
4. join
5. Threads Personalizados

# DCCajero Automático

### Contexto:

En el Departamento de Ciencias de la Computación (DCC), existe un fondo de curso administrado a través de un cajero automático virtual. Sin embargo, hay un pequeño problema: los ayudantes están realizando múltiples retiros al mismo tiempo, y sin una correcta sincronización del acceso a los fondos, podrían generarse errores en el saldo.

Tu misión es modelar este escenario utilizando threads en Python, demostrando cómo los retiros concurrentes pueden afectar el saldo del cajero y cómo puedes solucionarlo.

### Objetivos:

1. Simular un cajero automatico que maneje el fondo del curso.
2. Crear Threads personalizados para cada ayudante que intenta hacer retiros.
3. Implementar un hilo ue monitoree el saldo durante toda la ejecucion.
4. Utilizar locks para evitar problemas de concurrencia.

#### Clase CajeroAutomatico
Primero debemos simular nuestro cajero que administrara lo fondos, para esto crearemos la clase CajeroAutomatico y su metodo **retirar** para simular los retiros.

In [1]:
import threading
import time
import random

class CajeroAutomatico:

    def __init__(self, saldo_inicial):
        self.saldo = saldo_inicial
        # Creamos el lock para controlar el acceso al fondo
        self.retiro_lock = threading.Lock()

    def retirar(self, monto):
        # Usamos el lock para asegurar que no hayan retiros concurrentes
        with self.retiro_lock: 
            if monto > self.saldo: # Saldo insuficiente
                print(f"\nRetiro de {monto} no autorizado. Saldo insuficiente.")
                return False
            else:
                print(f"\nRetirando {monto}...")
                self.saldo -= monto # Efectuamos el retiro
                # Simulamos el tiempo de retiro con un tiempo aleatorio
                time.sleep(random.random())

#### Thread para simular retiros de cada ayudante
Creamos un thread personalizado para simular retiros de cada ayudante.

In [6]:
class RetiroThread(threading.Thread): # Heredamos de Thread
    def __init__(self, cajero, nombre, monto):
        super().__init__(name=nombre) # LLamamos a la clase padre, indicando el nombre
        self.cajero = cajero
        self.monto = monto

    # Sobreescribimos el metodo run para personalizar la ejecucion del thread
    def run(self):
        for i in range(random.randint(1, 10)): # numero aleatorio de retiros
            print(f"\n{self.name}, solicitando retiro de {self.monto}...")
            self.cajero.retirar(self.monto) # Realizamos el retiro
        print(f'\n{self.name} ha terminado de procesar los retiros.')

#### Thread de monitoreo de saldo
Creamos otro thread personalizado encargado de monitorear el saldo de la cuenta.

In [7]:
class MonitoreoThread(threading.Thread): # Heredamos de Thread
    def __init__(self, cajero):
        # ¿Por qué daemon?
        super().__init__(daemon=True)
        self.cajero = cajero

        # Esto es solo para controlar el thread dentro del notebook
        self.activo = True

    def run(self):
        while self.activo:
            print(f"\nMonitoreando saldo actual: {self.cajero.saldo}")
            time.sleep(2)  # Monitorea cada 2 segundos

    # Solo para detener el thread dentro del notebook
    # (en un programa real no se haría así, basta con cerrar el programa al ser daemon)
    def detener(self):
        self.activo = False

#### Ejecución del programa:
Ya que tenemos el cajero creado, y ambas clases de threads creadas, solo basta instanciar lo distintos threads y simular los retiros.

In [8]:
# Instanciamos el cajero
cajero = CajeroAutomatico(1000)

# Instanciamos los threads de retiro
agustin = RetiroThread(cajero, "Agustin", 10)
julio = RetiroThread(cajero, "Julio", 20)
francisca = RetiroThread(cajero, "Francisca", 30)
carlos = RetiroThread(cajero, "Carlos", 40)
diego = RetiroThread(cajero, "Diego", 50)

# Instanciamos e iniciamos el thread de monitoreo
monitoreo = MonitoreoThread(cajero)  # Daemon thread para monitoreo
monitoreo.start()

# Iniciamos los threads de retiro
agustin.start()
julio.start()
francisca.start()
carlos.start()
diego.start()

# Hacemo join a los threads de retiro
agustin.join()
julio.join()
francisca.join()
carlos.join()
diego.join()
# ¿Por que hacemos join? 

print("\nTodos los retiros han sido procesados.")
print(f"\nSaldo final: {cajero.saldo}")


monitoreo.detener()  # Detenemos el thread de monitoreo


Monitoreando saldo actual: 1000

Agustin, solicitando retiro de 10...

Retirando 10...

Julio, solicitando retiro de 20...

Francisca, solicitando retiro de 30...

Carlos, solicitando retiro de 40...

Diego, solicitando retiro de 50...

Agustin ha terminado de procesar los retiros.
Retirando 20...


Julio, solicitando retiro de 20...

Retirando 30...

Francisca, solicitando retiro de 30...

Retirando 40...

Monitoreando saldo actual: 900

Carlos ha terminado de procesar los retiros.
Retirando 50...


Diego, solicitando retiro de 50...
Retirando 20...


Julio, solicitando retiro de 20...

Retirando 30...

Francisca, solicitando retiro de 30...
Retirando 50...


Diego ha terminado de procesar los retiros.
Retirando 20...


Monitoreando saldo actual: 730

Julio, solicitando retiro de 20...

Retirando 30...

Francisca, solicitando retiro de 30...

Retirando 20...

Julio, solicitando retiro de 20...
Retirando 30...


Francisca ha terminado de procesar los retiros.
Retirando 20...


Monitor

# El DCCarrete

#### Contexto

Julio esta de cumpleaños y quiere hacer una gran fiesta para celebrar su nueva vuelta al sol. Pero esta preocupado con la organización de todo el evento, así que penso en una buena idea: "Simular toda la fiesta con threads" para poder ver que todo salga OK y que su casa no explote en el proceso.

### Clase `Fiesta`

Esta clase simula el entorno de la fiesta y gestiona los recursos y eventos compartidos por los participantes.

* **`__init__(self, nombre_fiesta="Fiesta en Casa")`**:
    * Inicializa la fiesta con un nombre.
    * `self.nombre_fiesta`: Almacena el nombre de la fiesta.
    * `self.lock_bano`: Un objeto `threading.Lock()` que representa el baño. Solo una `Persona` (thread) puede "usar" el baño a la vez.
    * `self.lock_pisco`: Un objeto `threading.Lock()` que representa la botella de pisco. Solo una `Persona` puede "servirse" pisco a la vez.
    * `self.evento_pizzas_listas`: Un objeto `threading.Event()` que indica si las pizzas han llegado y están listas para comer. Se inicializa como no señalado (False).
    * `self.evento_feliz_cumpleanos`: Un objeto `threading.Event()` que indica si es momento de cantar el feliz cumpleaños. Este evento detendrá las actividades normales de los participantes. Se inicializa como no señalado (False).
    * Imprime un mensaje indicando que la fiesta ha comenzado.

* **`llegan_las_pizzas(self)`**:
    * Simula la llegada de las pizzas.
    * Imprime un mensaje anunciando que las pizzas están listas.
    * Activa (señala) el evento `self.evento_pizzas_listas` llamando a su método `set()`. Esto notificará a las `Personas` que pueden empezar a comer pizza.

* **`momento_torta(self)`**:
    * Simula el momento de cantar el feliz cumpleaños.
    * Imprime un mensaje anunciando que es hora de la torta.
    * Activa (señala) el evento `self.evento_feliz_cumpleanos` llamando a su método `set()`. Esto notificará a las `Personas` que deben detener sus actividades y "prepararse" para cantar.

In [1]:
import threading
import time
import random

In [2]:
class Fiesta:
    def __init__(self, nombre_fiesta="Fiesta en Casa"):
        self.nombre_fiesta = nombre_fiesta
        self.lock_bano = threading.Lock()
        self.lock_pisco = threading.Lock()
        self.evento_pizzas_listas = threading.Event()
        self.evento_feliz_cumpleanos = threading.Event()
        print(f"¡Comienza la {self.nombre_fiesta}!")


    def llegan_las_pizzas(self):
        print("------------------------------------")
        print("🍕🍕🍕 ¡Llegaron las pizzas! 🍕🍕🍕")
        print("------------------------------------")
        self.evento_pizzas_listas.set()


    def momento_torta(self):
        print("\n------------------------------------")
        print("🎂 ¡Es hora de cantar el Feliz Cumpleaños! 🎂")
        print("------------------------------------")
        self.evento_feliz_cumpleanos.set()

### Clase `Persona(threading.Thread)`

Esta clase representa a cada persona que asiste a la fiesta. Cada instancia de `Persona` es un thread independiente que ejecuta sus acciones concurrentemente.

* **`__init__(self, nombre: str, fiesta: Fiesta, descanso_seg: int)`**:
    * Hereda de `threading.Thread`. Llama a `super().__init__(name=nombre)` para inicializar el thread y asignarle un nombre.
    * `self.nombre_persona`: Almacena el nombre de la persona para fácil acceso y uso en mensajes.
    * `self.fiesta`: Una referencia al objeto `Fiesta` al que asiste esta persona. Esto permite acceder a los locks y eventos compartidos.
    * `self.descanso_seg`: Un entero que representa cuántos segundos la persona descansará (usando `time.sleep()`) después de realizar ciertas actividades.
    * `self.daemon = True`: Configura el thread como daemon. Esto significa que el programa principal no esperará a que estos threads terminen si él mismo está finalizando. Los threads daemon se detienen abruptamente cuando todos los threads no-daemon han terminado.
    * `self.comio_pizza = False`: Un booleano para rastrear si la persona ya comió pizza, asegurando que solo lo haga una vez.

* **`run(self)`**:
    * Este método contiene la lógica principal que ejecuta el thread `Persona`. Se sobrescribe del método `run()` de la clase base `threading.Thread`.
    * Imprime un mensaje indicando que la persona ha llegado a la fiesta.
    * Inicia un bucle `while not self.fiesta.evento_feliz_cumpleanos.is_set()`: La persona continúa realizando actividades mientras no se haya activado el evento de "feliz cumpleaños".
        * **Comer Pizza**: Primero, verifica si el evento `evento_pizzas_listas` está activado (`is_set()`) y si la persona aún no ha comido (`not self.comio_pizza`).
            * Si ambas condiciones son verdaderas, imprime que está comiendo, simula un tiempo comiendo, marca `self.comio_pizza = True`, y usa `continue` para volver al inicio del bucle y reevaluar la situación (evitando hacer otra actividad inmediatamente después de comer).
        * **Servirse Pisco**: Si no comió pizza en esta iteración, intenta servirse pisco.
            * Utiliza `with self.fiesta.lock_pisco:` para adquirir el lock del pisco. Esto asegura que solo una persona pueda acceder a la "botella" a la vez. El `with` garantiza que el lock se libere automáticamente al salir del bloque.
            * Imprime mensajes indicando que tiene la botella, se está sirviendo y que se sirvió.
            * Simula un tiempo sirviéndose y luego descansa.
        * **Ir al Baño**: Después del pisco, intenta ir al baño.
            * Utiliza `with self.fiesta.lock_bano:` para adquirir el lock del baño.
            * Imprime mensajes indicando que está en el baño y que salió.
            * Simula un tiempo en el baño y luego descansa.
        * Imprime un mensaje genérico de que sigue disfrutando la fiesta antes de la siguiente iteración del bucle.
    * Una vez que el evento `evento_feliz_cumpleanos` es activado, el bucle `while` termina.
    * Imprime un mensaje final indicando que la persona va a cantar el cumpleaños, justo antes de que el método `run()` (y por lo tanto el thread) finalice.

In [3]:
class Persona(threading.Thread):
    def __init__(self, nombre: str, fiesta: Fiesta, descanso_seg: int):
        super().__init__(name=nombre) 
        self.nombre_persona = nombre 
        self.fiesta = fiesta
        self.descanso_seg = descanso_seg
        self.daemon = True
        self.comio_pizza = False

    def run(self):
        print(f"{self.nombre_persona} ha llegado a la {self.fiesta.nombre_fiesta}.")

        while not self.fiesta.evento_feliz_cumpleanos.is_set():
            if self.fiesta.evento_pizzas_listas.is_set() and not self.comio_pizza:
                print(f"{self.nombre_persona} está comiendo pizza. 🍕")
                time.sleep(self.descanso_seg)
                self.comio_pizza = True
                continue

            print(f"{self.nombre_persona} quiere un pisquito. 🍹")
            with self.fiesta.lock_pisco:
                print(f"LOCK PISCO: {self.nombre_persona} tiene la botella.")
                print(f"{self.nombre_persona} se está sirviendo pisco.")
                time.sleep(random.uniform(0.5, 1.5))
                print(f"{self.nombre_persona} se sirvió pisco. ¡Salud! 🥳")
            time.sleep(self.descanso_seg)

            print(f"{self.nombre_persona} quiere ir al baño. 🚻")
            with self.fiesta.lock_bano:
                print(f"LOCK BAÑO: {self.nombre_persona} está en el baño.")
                print(f"{self.nombre_persona} está usando el baño...")
                time.sleep(random.uniform(1, 3))
                print(f"{self.nombre_persona} salió del baño. ¡Qué alivio!😌")
            time.sleep(self.descanso_seg)            
            print(f"{self.nombre_persona} sigue disfrutando la fiesta...")

        print(f"🏃 {self.nombre_persona} va corriendo a cantar el cumpleaños!")

### Función `organizar_fiesta()`

Esta función orquesta toda la simulación de la fiesta.

* Crea una instancia de la clase `Fiesta`.
* Define una lista de nombres para los asistentes (`nombres_personas`) y un tiempo de descanso general.
* Itera sobre los nombres, creando una instancia de `Persona` para cada uno y almacenándolas en la lista `threads_personas`.
* Configura un `threading.Timer` llamado `timer_pizzas`. Este timer llamará al método `la_fiesta.llegan_las_pizzas()` después de `tiempo_pizzas` segundos. Inicia el timer.
* Itera sobre la lista `threads_personas` e inicia cada thread `Persona` llamando a su método `start()`. Esto hace que el método `run()` de cada persona comience a ejecutarse concurrentemente.
* Configura otro `threading.Timer` llamado `timer_cumpleanos`. Este timer llamará al método `la_fiesta.momento_torta()` después de `tiempo_para_torta` segundos. Este evento es el que finalmente detendrá el bucle principal en los threads `Persona`. Inicia el timer.
* Itera sobre la lista `threads_personas` y llama al método `join()` en cada una. Esto hace que el thread principal (el que ejecuta `organizar_fiesta()`) espere a que cada thread `Persona` termine su ejecución antes de continuar. Los threads `Persona` terminarán poco después de que se active `evento_feliz_cumpleanos`.
* Una vez que todos los threads `Persona` han terminado (es decir, todos los `join()` se han completado), imprime mensajes finales celebrando el cumpleaños y el éxito de la fiesta.

In [4]:
def organizar_fiesta():
    la_fiesta = Fiesta(nombre_fiesta="Fiesta Épica de Threads")
    nombres_personas = ["Julio", "Fran", "Diego", "Agustin", "Carlos"]
    threads_personas = []
    descanso_general = 2

    for nombre in nombres_personas:
        persona = Persona(nombre=nombre, fiesta=la_fiesta, descanso_seg=descanso_general)
        threads_personas.append(persona)

    tiempo_pizzas = 10
    timer_pizzas = threading.Timer(tiempo_pizzas, la_fiesta.llegan_las_pizzas)
    print(f"Las pizzas llegarán en {tiempo_pizzas} segundos...")
    timer_pizzas.start()

    for p_thread in threads_personas:
        p_thread.start()

    tiempo_para_torta = 35
    timer_cumpleanos = threading.Timer(tiempo_para_torta, la_fiesta.momento_torta)
    print(f"El feliz cumpleaños se cantará en {tiempo_para_torta} segundos...")
    timer_cumpleanos.start()

    for persona in threads_personas:
        persona.join()

    print("\n🎉🎉🎉 TODOS CANTANDO: ¡FELIZ CUMPLEAÑOS! 🎉🎉🎉")
    print(f"La {la_fiesta.nombre_fiesta} ha sido un éxito gracias a los Threads!")

## A Celebrar!
Con todo ya listo, porfín puedes correr la simulación del carrete!

In [5]:
organizar_fiesta()

¡Comienza la Fiesta Épica de Threads!
Las pizzas llegarán en 10 segundos...
Julio ha llegado a la Fiesta Épica de Threads.
Julio quiere un pisquito. 🍹
LOCK PISCO: Julio tiene la botella.
Julio se está sirviendo pisco.
Fran ha llegado a la Fiesta Épica de Threads.
Fran quiere un pisquito. 🍹
Diego ha llegado a la Fiesta Épica de Threads.
Diego quiere un pisquito. 🍹
Agustin ha llegado a la Fiesta Épica de Threads.
Agustin quiere un pisquito. 🍹
Carlos ha llegado a la Fiesta Épica de Threads.
Carlos quiere un pisquito. 🍹
El feliz cumpleaños se cantará en 35 segundos...
Julio se sirvió pisco. ¡Salud! 🥳
LOCK PISCO: Fran tiene la botella.
Fran se está sirviendo pisco.
Fran se sirvió pisco. ¡Salud! 🥳
LOCK PISCO: Diego tiene la botella.
Diego se está sirviendo pisco.
Julio quiere ir al baño. 🚻
LOCK BAÑO: Julio está en el baño.
Julio está usando el baño...
Diego se sirvió pisco. ¡Salud! 🥳
LOCK PISCO: Agustin tiene la botella.
Agustin se está sirviendo pisco.
Fran quiere ir al baño. 🚻
Agustin se s